# Hugging Face classification

resource: https://www.learnhuggingface.com/notebooks/hugging_face_text_classification_tutorial

setup: https://www.learnhuggingface.com/extras/setup

Need to use a GPU on Colb which can be done by going to Runtime and selecting change runtime type and selecting a GPU option


In [ ]:
# import necessary libraries
# Install dependencies (this is mostly for Google Colab, as the other dependences are available by default in Colab)
try:
  import datasets, evaluate, accelerate
  import gradio as gr
except ModuleNotFoundError:
  !pip install -U datasets evaluate accelerate gradio # -U stands for "upgrade" so we'll get the latest version by default
  import datasets, evaluate, accelerate
  import gradio as gr

import random

import numpy as np
import pandas as pd

import torch
import transformers

print(f"Using transformers version: {transformers.__version__}")
print(f"Using datasets version: {datasets.__version__}")
print(f"Using torch version: {torch.__version__}")

In [ ]:
# dataset for food and not_food to be used to make the classification model

from datasets import load_dataset

dataset = load_dataset("mrdbourke/learn_hf_food_not_food_image_captions")
dataset

In [ ]:
# What features exits?
dataset.column_names

In [ ]:
# Access training split

from datasets import load_dataset
# Re-load the dataset to ensure 'dataset' is a DatasetDict
dataset = load_dataset("mrdbourke/learn_hf_food_not_food_image_captions")

dataset["train"]

In [ ]:
# view both labels
dataset["train"].unique("label")

In [ ]:
# view some random samples
random_indices = random.sample(range(len(dataset["train"])), 5)
print(random_indices)

random_samples = dataset["train"][random_indices]
print(f"Random samples from dataset:\n")
for text, label in zip(random_samples["text"], random_samples["label"]):
  print(f"text: {text} | label: {label}")

In [ ]:
# check the count of each label
from collections import Counter

Counter(dataset["train"]["label"])

In [ ]:
# use random samples to create a dataset which creats a DataFrame
fnf_df = pd.DataFrame(dataset["train"])
fnf_df.sample(5)

In [ ]:
# creating a DataFrame with the counts of both labels
fnf_df["label"].value_counts()

Tokenize the text by using numbers for the machine to read and create a training split alongside a test split to train the machine and test it

In [ ]:
# map labels to numeric values

id2label = {0: "not_food", 1: "food"}
label2id = {"not_food": 0, "food": 1}

print(f"Label to ID mapping: {label2id}")
print(f"ID to Label mapping: {id2label}")

In [ ]:
# Create mappings programmatically from dataset

id2label = {idx: label for idx, label in enumerate(dataset["train"].unique("label")[::-1])}
label2id = label2id = {label: idx for idx, label in id2label.items()}

print(f"Label to ID mapping: {label2id}")
print(f"ID to Label mapping: {id2label}")

In [ ]:
# Turn labels into 0 for not_food and 1 for food
def map_labels_to_number(example):
  example["label"] = label2id[example["label"]]
  return example

example_sample = {"text": "This is a sentence about chicken.", "label": "food"}

# Test the function
map_labels_to_number(example_sample)

In [ ]:
# Map our dataset labels to numbers
dataset = dataset["train"].map(map_labels_to_number)
dataset[:5]

In [ ]:
dataset.shuffle()[:5]

split using dataset.train_test_split: https://huggingface.co/docs/datasets/v2.20.0/en/package_reference/main_classes#datasets.Dataset.train_test_split

In [ ]:
# Create train/test splits
dataset = dataset.train_test_split(test_size=0.2, seed=42) # seed isn't needed, without it you will get different splits each time you run the cell
dataset

In [ ]:
random_idx_train = random.randint(0, len(dataset["train"]))
random_sample_train = dataset["train"][random_idx_train]
random_sample_train

In [ ]:
random_idx_test = random.randint(0, len(dataset["test"]))
random_sample_test = dataset["test"][random_idx_test]
random_sample_test

all models: https://huggingface.co/models
tokenizer: https://huggingface.co/distilbert/distilbert-base-uncased
from_pretrained: https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes

In [ ]:
from transformers import AutoTokenizer

# uses fast tokenization which is supported by tokenziers library and implemented in Rust by default, and python if unavailable
tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name_or_path="distilbert/distilbert-base-uncased",
                                          use_fast=True)
tokenizer

In [ ]:
# Test out tokenizer
tokenizer("I love pizza")

In [ ]:
# Get the length of the vocabulary
length_of_tokenizer_vocab = len(tokenizer.vocab)
print(f"Length of tokenizer vocabulary: {length_of_tokenizer_vocab}")

# Get the maximum sequence length the tokenizer can handle
max_tokenizer_input_sequence_length = tokenizer.model_max_length
print(f"Max tokenizer input sequence length: {max_tokenizer_input_sequence_length}")

In [ ]:
tokenizer.vocab["pizza"]

In [ ]:
tokenizer("awawawawa")

In [ ]:
tokenizer.convert_ids_to_tokens(tokenizer("awawawawa").input_ids)

In [ ]:
# Try to tokenize an emoji
tokenizer.convert_ids_to_tokens(tokenizer("🍕").input_ids)

In [ ]:
# Get the first 5 items in the tokenizer vocab
sorted(tokenizer.vocab.items())[:5]

In [ ]:
import random
random.sample(sorted(tokenizer.vocab.items()), k=5)

In [ ]:
# tokenize given example text and return the tokenized text
def tokenize_text(examples):
    return tokenizer(examples["text"],
                     padding=True, # pad short sequences to longest sequence in the batch
                     truncation=True) # truncate long sequences to the maximum length the model can handle

In [ ]:
# test the function
example_sample_2 = {"text": "I love pizza", "label": 1}
tokenize_text(example_sample_2)

dataset.map: https://huggingface.co/docs/datasets/v2.20.0/en/package_reference/main_classes#datasets.Dataset.map

In [ ]:
# Map tokenize_text function to the dataset
tokenized_dataset = dataset.map(function=tokenize_text,
                                batched=True, # set batched=True to operate across batches of examples rather than only single examples for efficiency
                                batch_size=1000) # defaults to 1000, can be increased if you have a large dataset
tokenized_dataset

In [ ]:
# Get two samples from the tokenized dataset
train_tokenized_sample = tokenized_dataset["train"][0]
test_tokenized_sample = tokenized_dataset["test"][0]

for key in train_tokenized_sample.keys():
    print(f"[INFO] Key: {key}")
    print(f"Train sample: {train_tokenized_sample[key]}")
    print(f"Test sample: {test_tokenized_sample[key]}")
    print("")

Use an evaluation metric to see how the model is performing by checking for accuracy (how many it gets correct over the number of total trials ran)

In [ ]:
import evaluate
import numpy as np
from typing import Tuple

accuracy_metric = evaluate.load("accuracy")

def compute_accuracy(predictions_and_labels: Tuple[np.array, np.array]):
  predictions, labels = predictions_and_labels
  # get highest prediction probability of each prediction if predictions are probabilities
  if len(predictions.shape) >= 2:
    predictions = np.argmax(predictions, axis=1)
  return accuracy_metric.compute(predictions=predictions, references=labels)

In [ ]:
# Create example list of predictions and labels
example_predictions_all_correct = np.array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0])
example_predictions_one_wrong = np.array([1, 0, 0, 0, 0, 0, 0, 0, 0, 0])
example_labels = np.array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

print(f"Accuracy when all predictions are correct: {compute_accuracy((example_predictions_all_correct, example_labels))}")
print(f"Accuracy when one prediction is wrong: {compute_accuracy((example_predictions_one_wrong, example_labels))}")

Now set up a model for training:
define model: https://huggingface.co/docs/transformers/en/model_doc/auto#transformers.AutoModelForSequenceClassification

define training arguments: https://huggingface.co/docs/transformers/en/main_classes/trainer#transformers.TrainingArguments

pass arguments as instance of: https://huggingface.co/docs/transformers/en/main_classes/trainer

call Trainer.train(): https://huggingface.co/docs/transformers/v4.40.2/en/main_classes/trainer#transformers.Trainer.train

evaluate model and share

In [ ]:
from transformers import AutoModelForSequenceClassification

# Setup model for fine-tuning with classification head (top layers of network)
model = AutoModelForSequenceClassification.from_pretrained(
    pretrained_model_name_or_path="distilbert/distilbert-base-uncased",
    num_labels=2, # food and not_food
    id2label=id2label, # mappings from class IDs to the class labels
    label2id=label2id
)

In [ ]:
model

In [ ]:
# Count the parameters of a PyTorch model
def count_params(model):
    trainable_parameters = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_parameters = sum(p.numel() for p in model.parameters())
    return {"trainable_parameters": trainable_parameters, "total_parameters": total_parameters}

count_params(model)

parameters for model: https://huggingface.co/docs/transformers/en/model_doc/auto#transformers.AutoModelForSequenceClassification

In [ ]:
# create model output directory
from pathlib import Path

# create models directory
models_dir = Path("models")
models_dir.mkdir(exist_ok=True)

# create model save name
model_save_name = "learn_hf_food_not_food_text_classifier-distilbert-base-uncased"

# create model save path
model_save_dir = Path(models_dir, model_save_name)

model_save_dir

training arguments: https://huggingface.co/docs/transformers/en/main_classes/trainer#transformers.TrainingArguments

In [ ]:
# define training arguments with transformers.TrainingArguments
from transformers import TrainingArguments

print(f"[INFO] Saving model checkpoints to: {model_save_dir}")

# Create training arguments
training_args = TrainingArguments(
    output_dir=model_save_dir,
    learning_rate=0.0001,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=3, # limit the total amount of save checkpoints
    use_cpu=False,
    seed=42, # set to 42 by default for reproducibility
    load_best_model_at_end=True,
    logging_strategy="epoch", # log training results every epoch
)

transformers.Trainer: https://huggingface.co/docs/transformers/en/main_classes/trainer

In [ ]:
# pass training arguments into instance of transformers.Trainer
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    compute_metrics=compute_accuracy
)

Trainer.train(): https://huggingface.co/docs/transformers/v4.40.2/en/main_classes/trainer#transformers.Trainer.train

In [ ]:
results = trainer.train()

In [ ]:
# inspect training metrics
for key, value in results.metrics.items():
    print(f"{key}: {value}")

save model: https://huggingface.co/docs/transformers/en/main_classes/trainer#transformers.Trainer.save_model

In [ ]:
# Save model
print(f"[INFO] Saving model to {model_save_dir}")
trainer.save_model(output_dir=model_save_dir)

In [ ]:
# get training history
trainer_history_all = trainer.state.log_history
trainer_history_metrics = trainer_history_all[:-1] # get everything except the training time metrics
trainer_history_training_time = trainer_history_all[-1] # this is the same value as results.metrics from above

# view the first 3 metrics from the training history
trainer_history_metrics[:3]

In [ ]:
import pprint # import pretty print

# training and evaluation metrics
trainer_history_training_set = []
trainer_history_eval_set = []

# loop through metrics and filter for training and eval metrics
for item in trainer_history_metrics:
    item_keys = list(item.keys())
    # check if "eval" is in the keys of the item
    if any("eval" in item for item in item_keys):
        trainer_history_eval_set.append(item)
    else:
        trainer_history_training_set.append(item)

# show the first two items in each metric set
print(f"[INFO] First two items in training set:")
pprint.pprint(trainer_history_training_set[:2])

print(f"\n[INFO] First two items in evaluation set:")
pprint.pprint(trainer_history_eval_set[:2])

In [ ]:
# dataFrames for the training and evaluation metrics
trainer_history_training_df = pd.DataFrame(trainer_history_training_set)
trainer_history_eval_df = pd.DataFrame(trainer_history_eval_set)

trainer_history_training_df.head()

In [ ]:
# evaluation metric
trainer_history_eval_df.head()

In [ ]:
# plot training and evaluation loss
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.plot(trainer_history_training_df["epoch"], trainer_history_training_df["loss"], label="Training loss")
plt.plot(trainer_history_eval_df["epoch"], trainer_history_eval_df["eval_loss"], label="Evaluation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Text classification with DistilBert training and evaluation loss over time")
plt.legend()
plt.show()

make predictions with: https://huggingface.co/docs/transformers/v4.40.2/en/main_classes/trainer#transformers.Trainer.predict

In [ ]:
# perform predictions on the test set
predictions_all = trainer.predict(tokenized_dataset["test"])
prediction_values = predictions_all.predictions
prediction_metrics = predictions_all.metrics

print(f"[INFO] Prediction metrics on the test data:")
prediction_metrics

torch.softmax: https://docs.pytorch.org/docs/2.12/generated/torch.nn.Softmax.html

torch.argmax: https://docs.pytorch.org/docs/2.12/generated/torch.argmax.html

np.argmax: https://numpy.org/doc/stable/reference/generated/numpy.argmax.html

accuracy score: https://scikit-learn.org/stable/modules/generated/sklearn.metrics.accuracy_score.html

In [ ]:
import torch
from sklearn.metrics import accuracy_score

# get prediction probabilities
pred_probs = torch.softmax(torch.tensor(prediction_values), dim=1)

# get the predicted labels
pred_labels = torch.argmax(pred_probs, dim=1)

# get the true labels
true_labels = dataset["test"]["label"]

# compare predicted labels to true labels to get the test accuracy
test_accuracy = accuracy_score(y_true=true_labels,
                               y_pred=pred_labels)

print(f"[INFO] Test accuracy: {test_accuracy*100}%")

In [ ]:
# make a DataFrame of test predictions
test_predictions_df = pd.DataFrame({
    "text": dataset["test"]["text"],
    "true_label": true_labels,
    "pred_label": pred_labels,
    "pred_prob": torch.max(pred_probs, dim=1).values
})

test_predictions_df.head()

## How to Improve Model Prediction Accuracy

To make the model predict more accurately and with higher confidence, especially in cases where it's uncertain (like the 'tractor' example), here are some common strategies:

1.  **More Diverse and Labeled Data**: The most impactful improvement usually comes from training on a larger and more diverse dataset. If the model hasn't seen enough examples of 'not_food' items that are visually distinct from 'food' items, it might struggle to generalize.
    *   **Action**: Collect more image captions for both 'food' and 'not_food' categories, focusing on edge cases or ambiguous examples.

2.  **Data Augmentation**: Even with a limited dataset, you can create synthetic variations of your existing data to increase its effective size. For text, this could involve:
    *   **Action**: Using techniques like synonym replacement, random insertion/deletion/swapping of words, or back-translation (translate to another language and back).

3.  **Hyperparameter Tuning**: Experimenting with the training parameters can sometimes yield better results.
    *   **Action**: Try different `learning_rate` values (e.g., smaller learning rates like `1e-5`, `5e-6`), more `num_train_epochs`, or adjusting `per_device_train_batch_size`.

4.  **Different Model Architectures**: While `DistilBERT` is efficient, a larger model might capture more nuanced features.
    *   **Action**: Experiment with other pre-trained models from the Hugging Face Hub, such as `BERT-base-uncased`, `RoBERTa-base`, or even larger variants if computational resources allow.

5.  **Fine-tuning Longer**: Sometimes models need more training time to fully converge and learn the intricate patterns in the data.
    *   **Action**: Increase the `num_train_epochs` (e.g., from 10 to 20 or more) and monitor the training and evaluation loss to ensure it's not overfitting.

6.  **Addressing Class Imbalance**: Although your dataset is balanced (125 food, 125 not_food), if the new data you introduce becomes imbalanced, the model might favor the majority class.
    *   **Action**: If future datasets are imbalanced, consider techniques like oversampling the minority class, undersampling the majority class, or using class weights during training.

7.  **Error Analysis**: Manually inspect more examples where the model makes incorrect or low-confidence predictions. This can help you identify patterns in the errors and guide your data collection or augmentation efforts.
    *   **Action**: Review the `test_predictions_df` for texts where `pred_label` is wrong, or `pred_prob` is close to 0.5, to understand why the model struggled.

In [ ]:
local_model_path = "/content/models/learn_hf_food_not_food_text_classifier-distilbert-base-uncased"

MAKE PREDICTIONS WITH

pipeline mode for prediction: https://huggingface.co/docs/transformers/v4.42.0/en/main_classes/pipelines#pipelines



In [ ]:
def set_device():
    if torch.cuda.is_available():
        device = torch.device("cuda")
    elif torch.backends.mps.is_available() and torch.backends.mps.is_built():
        device = torch.device("mps")
    else:
        device = torch.device("cpu")
    return device

DEVICE = set_device()
print(f"[INFO] Using device: {DEVICE}")

In [ ]:
trainer.train()
trainer.save_model(local_model_path)
tokenizer.save_pretrained(local_model_path)

In [ ]:
import torch
from transformers import pipeline

BATCH_SIZE = 32

# create an instance of transformers.pipeline
food_not_food_classifier = pipeline(task="text-classification",
                                    model=local_model_path,
                                    device=DEVICE,
                                    top_k=1,
                                    batch_size=BATCH_SIZE)

food_not_food_classifier

In [ ]:
# test model on some example text
sample_text_food = "A delicious photo of a plate of scrambled eggs"
food_not_food_classifier(sample_text_food)

In [ ]:
# test model on some example text
sample_text_food = "Ben is a fat chud that eats pizza and cookies everyday"
food_not_food_classifier(sample_text_food)

In [ ]:
# test model on some example text
sample_text_not_food = "A delicious photo of a plate"
food_not_food_classifier(sample_text_not_food)

In [ ]:
# test model on some example text
sample_text_not_food = "Driving in an infinity is so tough"
food_not_food_classifier(sample_text_not_food)

In [ ]:
# Pass in random text to the model
food_not_food_classifier("cvnhertiejhwgdjshdfgh394587")

In [ ]:
BATCH_SIZE = 32

food_not_food_classifier = pipeline(task="text-classification",
                                    model=local_model_path,
                                    batch_size=BATCH_SIZE,
                                    device=DEVICE)

In [ ]:
"""
seems to work pretty well and in the case of labeling something as food even though it's not, it's accuracy is lower which makes sense
"""
# Create a list of sentences to make predictions on
sentences = [
    "I whipped up a fresh batch of code, but it seems to have a syntax error.",
    "The new software is definitely a spicy upgrade, taking some time to get used to.",
    "Her social media post was the perfect recipe for a viral sensation.",
    "He served up a rebuttal full of facts, leaving his opponent speechless.",
    "The team needs to simmer down a bit before tackling the next challenge.",
    "The presentation was a delicious blend of humor and information, keeping the audience engaged.",
    "My favoruite food is chicken!",
    "protein pasta with noodles is fire"
]

food_not_food_classifier(sentences)

In [ ]:
# check timing
import time

# Create 800 sentences
sentences_800 = sentences * 100

# Time how long it takes to make predictions on all sentences (one at a time)
print(f"[INFO] Number of sentences: {len(sentences_800)}")
start_time_one_at_a_time = time.time()
for sentence in sentences_800:
    food_not_food_classifier(sentence)
end_time_one_at_a_time = time.time()

print(f"[INFO] Time taken for one at a time prediction: {end_time_one_at_a_time - start_time_one_at_a_time} seconds")
print(f"[INFO] Avg inference time per sentence: {(end_time_one_at_a_time - start_time_one_at_a_time) / len(sentences_800)} seconds")

In [ ]:
# testing the timing with batching
for i in [10, 100, 1000, 10_000]:
    sentences_big = sentences * i
    print(f"[INFO] Number of sentences: {len(sentences_big)}")

    start_time = time.time()
    # Predict on all sentences in batches
    food_not_food_classifier(sentences_big)
    end_time = time.time()

    print(f"[INFO] Inference time for {len(sentences_big)} sentences: {round(end_time - start_time, 5)} seconds.")
    print(f"[INFO] Avg inference time per sentence: {round((end_time - start_time) / len(sentences_big), 8)} seconds.")
    print()

from above we can see that batching allows to run more samples at once and runs faster than individually going through samples